# CTR Prediction with FuxiCTR

基于 FuxiCTR 2.3.10，在 Criteo 500K 子集上复现 LR、FM、DeepFM，并比较 DeepFM 在不同 Embedding 维度下的表现。

**运行环境：Google Colab（建议启用 GPU）**

本 Notebook 按照实际实验流程整理。训练单元格会重新训练模型；如果只想查看已完成的实验结果，可以跳过训练单元格，直接查看最后的结果汇总。

## 1. Environment Setup

在 Colab 中选择 `Runtime → Change runtime type → GPU`。

先检查运行环境，再上传你之前下载的 **FuxiCTR 2.3.10 源码 ZIP**。ZIP 解压后应包含 `FuxiCTR-main/requirements.txt`。

In [ ]:
!nvidia-smi

In [ ]:
import os
import sys
import json
import zipfile
import shutil
from pathlib import Path

import torch
import psutil

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

print("System RAM:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")
print("Free disk:", round(shutil.disk_usage("/content").free / 1024**3, 2), "GB")

上传 FuxiCTR 2.3.10 ZIP。这里沿用本次实验使用的源码包，不会把 FuxiCTR 官方源码复制进自己的 GitHub 仓库。

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if not zip_names:
    raise ValueError("请上传 FuxiCTR 源码 ZIP 文件。")

fuxictr_zip = zip_names[0]

with zipfile.ZipFile(fuxictr_zip, "r") as zf:
    zf.extractall("/content")

PROJECT_ROOT = Path("/content/FuxiCTR-main")

if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "没有找到 /content/FuxiCTR-main/requirements.txt。"
        "请确认 ZIP 解压后的顶层目录名称为 FuxiCTR-main。"
    )

print("FuxiCTR source:", PROJECT_ROOT)

In [ ]:
%cd /content/FuxiCTR-main

!pip install -r requirements.txt
!pip install -e . --no-deps

In [ ]:
import fuxictr
from importlib.metadata import version

print("FuxiCTR version:", version("fuxictr"))
print("FuxiCTR package:", fuxictr.__file__)

assert version("fuxictr") == "2.3.10", "请使用本项目实验对应的 FuxiCTR 2.3.10。"

## 2. Dataset Preparation

数据来源：[`reczoo/Criteo_x1`](https://huggingface.co/datasets/reczoo/Criteo_x1)。

从数据集的 train、valid、test 文件中分别抽取 350,000、100,000、50,000 条样本，组成 Criteo 500K 实验数据。

本实验固定随机种子为 `20261115`。抽样分两步完成：先按数据块抽取候选样本，再从候选样本中抽取指定数量。

In [ ]:
DATASET_DIR = Path("/content/datasets/criteo_x1")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ZIP = DATASET_DIR / "Criteo_x1.zip"

!curl -L --retry 5 -C - -o /content/datasets/criteo_x1/Criteo_x1.zip "https://huggingface.co/datasets/reczoo/Criteo_x1/resolve/main/Criteo_x1.zip"

assert DATASET_ZIP.exists() and DATASET_ZIP.stat().st_size > 0
print("Downloaded:", DATASET_ZIP)
print("Size (GB):", round(DATASET_ZIP.stat().st_size / 1024**3, 3))

In [ ]:
# 可选：核对下载文件的 SHA256
!sha256sum /content/datasets/criteo_x1/Criteo_x1.zip

with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
    csv_files = set(zf.namelist())

required_files = {"train.csv", "valid.csv", "test.csv"}
if not required_files.issubset(csv_files):
    raise FileNotFoundError(f"数据压缩包缺少文件：{required_files - csv_files}")

print("Archive check passed.")

先抽查原始数据字段和标签分布。这里每个 split 只读取前 100,000 行用于快速检查，不改变后续正式抽样过程。

In [ ]:
import pandas as pd
import numpy as np
import io

numeric_cols = [f"I{i}" for i in range(1, 14)]
categorical_cols = [f"C{i}" for i in range(1, 27)]
expected_cols = ["label"] + numeric_cols + categorical_cols

with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
    for split in ["train", "valid", "test"]:
        with zf.open(f"{split}.csv") as raw:
            df_head = pd.read_csv(raw, nrows=100_000)

        df_head.columns = df_head.columns.str.strip()

        print(f"\n{split}:")
        print("Shape (first 100k rows):", df_head.shape)
        print("Column count:", len(df_head.columns))
        print("Positive rate:", round(df_head["label"].mean(), 5))
        print("Missing values:", int(df_head.isna().sum().sum()))

        assert list(df_head.columns) == expected_cols

### 2.1 Sample 500K rows

以下抽样代码沿用本次实验实际使用的方法和参数。输出文件保存在 `/content/datasets/criteo_x1/sample_500k/`。

In [ ]:
import gc

SAMPLE_DIR = DATASET_DIR / "sample_500k"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

sample_targets = {
    "train": 350_000,
    "valid": 100_000,
    "test": 50_000
}

SEED = 20261115
SAMPLE_FRAC = 0.014

dtype_map = {"label": "int8"}
dtype_map.update({col: "float32" for col in numeric_cols})
dtype_map.update({col: "string" for col in categorical_cols})

with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
    for split, target_size in sample_targets.items():
        print(f"\n========== {split} ==========")

        sampled_parts = []

        with zf.open(f"{split}.csv") as raw:
            reader = pd.read_csv(
                raw,
                chunksize=100_000,
                dtype=dtype_map,
                skipinitialspace=True
            )

            for chunk_idx, chunk in enumerate(reader):
                chunk.columns = chunk.columns.str.strip()

                if chunk_idx == 0:
                    assert list(chunk.columns) == expected_cols

                part = chunk.sample(
                    frac=SAMPLE_FRAC,
                    random_state=SEED + chunk_idx
                )
                sampled_parts.append(part)
                del chunk

        candidates = pd.concat(sampled_parts, ignore_index=True)
        del sampled_parts
        gc.collect()

        print("Candidate samples:", len(candidates))

        if len(candidates) < target_size:
            raise RuntimeError(
                f"{split} 候选样本不足：{len(candidates)} < {target_size}。"
                "请检查数据文件或调整 SAMPLE_FRAC。"
            )

        final_sample = candidates.sample(
            n=target_size,
            random_state=SEED
        ).reset_index(drop=True)

        output_path = SAMPLE_DIR / f"{split}.csv"
        final_sample.to_csv(output_path, index=False)

        print("Final samples:", len(final_sample))
        print("Positive rate:", round(final_sample["label"].mean(), 5))
        print("Missing values:", int(final_sample.isna().sum().sum()))
        print("Saved to:", output_path)

        del candidates, final_sample
        gc.collect()

print("\nCriteo 500K sampling completed.")

## 3. Feature Processing

使用 FuxiCTR `FeatureProcessor` 将原始 CSV 转换为 Parquet，并生成 FeatureMap。

本实验设置 `min_categr_count=10`。数值特征为 I1–I13，类别特征为 C1–C26。

**数据划分说明：** FuxiCTR 2.3.10 的 `build_dataset` 使用训练集拟合特征处理器，再将验证集和测试集转换为相同的特征空间。可参考 [build_dataset.py](https://github.com/reczoo/FuxiCTR/blob/v2.3.10/fuxictr/preprocess/build_dataset.py)。

In [ ]:
import yaml
import logging

from fuxictr.preprocess import FeatureProcessor, build_dataset
from fuxictr.utils import load_dataset_config, set_logger, print_to_json

DEMO_DIR = PROJECT_ROOT / "demo"
DEMO_CONFIG_DIR = DEMO_DIR / "config" / "criteo_500k_config"
DEMO_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "criteo_500k"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / DATASET_ID

preprocess_config = {
    DATASET_ID: {
        "data_root": "../data/",
        "data_format": "csv",
        "train_data": str(SAMPLE_DIR / "train.csv"),
        "valid_data": str(SAMPLE_DIR / "valid.csv"),
        "test_data": str(SAMPLE_DIR / "test.csv"),
        "min_categr_count": 10,
        "feature_cols": [
            {
                "name": numeric_cols,
                "active": True,
                "dtype": "float",
                "type": "numeric"
            },
            {
                "name": categorical_cols,
                "active": True,
                "dtype": "str",
                "type": "categorical"
            }
        ],
        "label_col": {"name": "label", "dtype": "float"}
    }
}

preprocess_config_path = DEMO_CONFIG_DIR / "dataset_config.yaml"
preprocess_config_path.write_text(
    yaml.safe_dump(preprocess_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)

print("Saved:", preprocess_config_path)

In [ ]:
# 若需要强制重新预处理，将此变量改为 True。
REBUILD_DATASET = False

os.chdir(DEMO_DIR)

params = load_dataset_config(str(DEMO_CONFIG_DIR), DATASET_ID)
set_logger(params)
logging.info("Params: " + print_to_json(params))

feature_encoder = FeatureProcessor(
    feature_cols=params["feature_cols"],
    label_col=params["label_col"],
    dataset_id=DATASET_ID,
    data_root=params["data_root"]
)

required_outputs = [
    PROCESSED_DATA_DIR / "feature_map.json",
    PROCESSED_DATA_DIR / "train.parquet",
    PROCESSED_DATA_DIR / "valid.parquet",
    PROCESSED_DATA_DIR / "test.parquet"
]

if REBUILD_DATASET and PROCESSED_DATA_DIR.exists():
    shutil.rmtree(PROCESSED_DATA_DIR)

if not all(path.exists() for path in required_outputs):
    if PROCESSED_DATA_DIR.exists():
        shutil.rmtree(PROCESSED_DATA_DIR)

    build_dataset(
        feature_encoder,
        train_data=params["train_data"],
        valid_data=params["valid_data"],
        test_data=params["test_data"],
        data_block_size=params.get("data_block_size", 0)
    )
    print("Preprocessing completed.")
else:
    print("Existing processed dataset found; skip rebuilding.")
    print("Set REBUILD_DATASET=True if you need to rebuild it.")

In [ ]:
# Check processed data and FeatureMap
data_dir = PROCESSED_DATA_DIR

for split, expected_rows in [
    ("train", 350_000),
    ("valid", 100_000),
    ("test", 50_000)
]:
    df = pd.read_parquet(data_dir / f"{split}.parquet")

    print(f"\n{split}:")
    print("Shape:", df.shape)
    print("Missing values:", int(df.isna().sum().sum()))
    print("Positive rate:", round(float(df["label"].mean()), 5))

    assert len(df) == expected_rows
    assert df.isna().sum().sum() == 0

with open(data_dir / "feature_map.json", "r", encoding="utf-8") as f:
    feature_map = json.load(f)

print("\nFeatureMap")
print("num_fields:", feature_map.get("num_fields"))
print("total_features:", feature_map.get("total_features"))
print("input_length:", feature_map.get("input_length"))
print("feature count:", len(feature_map.get("features", {})))

assert feature_map.get("num_fields") == 39
assert feature_map.get("total_features") == 361285

## 4. Model Configuration

统一主要训练设置：Adam、Learning Rate 0.001、Batch Size 1024、最大 Epoch 10、Early Stopping Patience 2、Random Seed 20261115，监控验证集 AUC。

FM 使用成功完成训练和评测的 `FM_criteo_500k_rerun` 实验 ID。第一次未完整结束的 FM 运行不纳入正式结果。

In [ ]:
PROJECT_ROOT = Path("/content/FuxiCTR-main")
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "criteo_500k"
CONFIG_DIR = EXPERIMENT_DIR / "config"
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

dataset_config = {
    "criteo_500k": {
        "data_root": str(PROJECT_ROOT / "data") + "/",
        "data_format": "parquet",
        "train_data": str(PROCESSED_DATA_DIR / "train.parquet"),
        "valid_data": str(PROCESSED_DATA_DIR / "valid.parquet"),
        "test_data": str(PROCESSED_DATA_DIR / "test.parquet")
    }
}

(CONFIG_DIR / "dataset_config.yaml").write_text(
    yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)

base_config = {
    "num_workers": 2,
    "verbose": 1,
    "early_stop_patience": 2,
    "pickle_feature_encoder": True,
    "save_best_only": True,
    "eval_steps": None,
    "debug_mode": False
}

def common_model_config(model_name, model_root, **extra):
    config = {
        "model": model_name,
        "model_root": str(model_root) + "/",
        "dataset_id": "criteo_500k",
        "loss": "binary_crossentropy",
        "metrics": ["logloss", "AUC"],
        "task": "binary_classification",
        "optimizer": "adam",
        "learning_rate": 0.001,
        "batch_size": 1024,
        "epochs": 10,
        "shuffle": True,
        "seed": 20261115,
        "monitor": "AUC",
        "monitor_mode": "max"
    }
    config.update(extra)
    return config

model_config = {"Base": base_config}

model_config["LR_criteo_500k"] = common_model_config(
    "LR",
    CHECKPOINT_DIR / "LR",
    regularizer=0
)

model_config["FM_criteo_500k_rerun"] = common_model_config(
    "FM",
    CHECKPOINT_DIR / "FM_rerun",
    regularizer=0,
    embedding_dim=16
)

deepfm_base = common_model_config(
    "DeepFM",
    CHECKPOINT_DIR / "DeepFM",
    embedding_dim=16,
    hidden_units=[64, 32],
    hidden_activations="relu",
    net_regularizer=0,
    embedding_regularizer=0,
    batch_norm=False,
    net_dropout=0
)
model_config["DeepFM_criteo_500k"] = deepfm_base

for dim in [8, 32]:
    expid = f"DeepFM_criteo_500k_emb{dim}"
    model_config[expid] = common_model_config(
        "DeepFM",
        CHECKPOINT_DIR / f"DeepFM_emb{dim}",
        embedding_dim=dim,
        hidden_units=[64, 32],
        hidden_activations="relu",
        net_regularizer=0,
        embedding_regularizer=0,
        batch_norm=False,
        net_dropout=0
    )

(CONFIG_DIR / "model_config.yaml").write_text(
    yaml.safe_dump(model_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)

print("Dataset config:", CONFIG_DIR / "dataset_config.yaml")
print("Model config:", CONFIG_DIR / "model_config.yaml")

In [ ]:
from fuxictr.utils import load_config

expids = [
    "LR_criteo_500k",
    "FM_criteo_500k_rerun",
    "DeepFM_criteo_500k",
    "DeepFM_criteo_500k_emb8",
    "DeepFM_criteo_500k_emb32"
]

for expid in expids:
    params = load_config(str(CONFIG_DIR), expid)
    print(
        expid,
        "| model:", params["model"],
        "| embedding_dim:", params.get("embedding_dim", "N/A"),
        "| seed:", params["seed"]
    )

## 5. Baseline Model Comparison

按顺序运行以下三个模型。每个训练单元格都会调用 FuxiCTR 对应模型目录中的官方 `run_expid.py`。

### 5.1 LR

In [ ]:
!cd /content/FuxiCTR-main/model_zoo/LR && python run_expid.py --config /content/FuxiCTR-main/experiments/criteo_500k/config --expid LR_criteo_500k --gpu 0

### 5.2 FM

In [ ]:
!cd /content/FuxiCTR-main/model_zoo/FM && python run_expid.py --config /content/FuxiCTR-main/experiments/criteo_500k/config --expid FM_criteo_500k_rerun --gpu 0

### 5.3 DeepFM (Embedding Dim = 16)

In [ ]:
!cd /content/FuxiCTR-main/model_zoo/DeepFM/DeepFM_torch && python run_expid.py --config /content/FuxiCTR-main/experiments/criteo_500k/config --expid DeepFM_criteo_500k --gpu 0

## 6. Embedding Dimension Experiment

固定 DeepFM 其他主要配置，仅改变 Embedding Dim。16 维使用上一节的基础 DeepFM 结果；下面分别运行 8 维和 32 维。

### 6.1 Embedding Dim = 8

In [ ]:
!cd /content/FuxiCTR-main/model_zoo/DeepFM/DeepFM_torch && python run_expid.py --config /content/FuxiCTR-main/experiments/criteo_500k/config --expid DeepFM_criteo_500k_emb8 --gpu 0

### 6.2 Embedding Dim = 32

In [ ]:
!cd /content/FuxiCTR-main/model_zoo/DeepFM/DeepFM_torch && python run_expid.py --config /content/FuxiCTR-main/experiments/criteo_500k/config --expid DeepFM_criteo_500k_emb32 --gpu 0

## 7. Results Summary

以下结果来自本次已完成实验的 FuxiCTR 日志。重新运行训练后，应以新日志核对并更新此处结果。

In [ ]:
baseline_results = pd.DataFrame([
    {
        "Model": "LR",
        "Parameters": 361286,
        "Actual Epochs": 10,
        "Best Epoch": 9,
        "Validation AUC": 0.763905,
        "Validation LogLoss": 0.481453,
        "Test AUC": 0.761254,
        "Test LogLoss": 0.483803
    },
    {
        "Model": "FM",
        "Parameters": 6141846,
        "Actual Epochs": 5,
        "Best Epoch": 3,
        "Validation AUC": 0.760212,
        "Validation LogLoss": 0.488086,
        "Test AUC": 0.758528,
        "Test LogLoss": 0.490112
    },
    {
        "Model": "DeepFM (dim=16)",
        "Parameters": 6183959,
        "Actual Epochs": 3,
        "Best Epoch": 1,
        "Validation AUC": 0.773412,
        "Validation LogLoss": 0.473758,
        "Test AUC": 0.770871,
        "Test LogLoss": 0.475868
    }
])

baseline_results

In [ ]:
embedding_results = pd.DataFrame([
    {
        "Embedding Dim": 8,
        "Parameters": 3273711,
        "Actual Epochs": 3,
        "Best Epoch": 1,
        "Validation AUC": 0.773059,
        "Validation LogLoss": 0.474296,
        "Test AUC": 0.769889,
        "Test LogLoss": 0.477265
    },
    {
        "Embedding Dim": 16,
        "Parameters": 6183959,
        "Actual Epochs": 3,
        "Best Epoch": 1,
        "Validation AUC": 0.773412,
        "Validation LogLoss": 0.473758,
        "Test AUC": 0.770871,
        "Test LogLoss": 0.475868
    },
    {
        "Embedding Dim": 32,
        "Parameters": 12004455,
        "Actual Epochs": 3,
        "Best Epoch": 1,
        "Validation AUC": 0.773931,
        "Validation LogLoss": 0.473322,
        "Test AUC": 0.770693,
        "Test LogLoss": 0.475698
    }
])

embedding_results

### Notes

- `Actual Epochs` 是实际训练轮数；`Best Epoch` 是验证集 AUC 最佳的检查点所在轮数。
- 模型选择依据为 Validation AUC，测试集指标用于最终报告。
- 当前结果来自单次随机种子实验，微小指标差异尚未经过多随机种子验证。
- 原始数据、抽样 CSV、Parquet、模型权重和运行日志不应直接提交到 GitHub。
- 本 Notebook 在整理时清空了原始执行输出；准备公开前，建议在新的 Colab Runtime 中按顺序重新运行一次，确认从下载到结果汇总的流程可复现。